In [3]:
import openml
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

def load_openml_dataset(dataset_id):
    """
    Fetches dataset from OpenML, handles NaN and encoding.
    Returns: X (numpy array), y (numpy array)
    """
    print(f"Downloading dataset ID {dataset_id}...")
    
    # Fetch dataset with 'dataframe' format to inspect types easily
    dataset = openml.datasets.get_dataset(dataset_id)
    
    try:
        # OpenML python API can sometimes return sparse frames
        X, y, categorical_indicator, attribute_names = dataset.get_data(
            target=dataset.default_target_attribute,
            dataset_format='dataframe'
        )
    except Exception:
        # Fallback for older openml versions
        X, y, categorical_indicator, attribute_names = dataset.get_data(
            target=dataset.default_target_attribute
        )
    
    # 1. Convert Sparse to Dense if necessary
    # Gisette (ID 41138) often loads as a SparseDataFrame which causes issues with Imputers
    if hasattr(X, "sparse"):
        # Pandas < 1.0 style or specific SparseDtype
        try:
            X = X.sparse.to_dense()
        except:
            pass
    
    # Ensure X is a DataFrame for type separation
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X, columns=attribute_names)

    # 2. Split Numeric and Categorical
    # We handle them separately to apply different imputation strategies
    num_cols = X.select_dtypes(include=[np.number]).columns
    cat_cols = X.select_dtypes(exclude=[np.number]).columns
    
    # Extract values as numpy arrays to avoid "Columns must be same length as key" errors
    # This happens when Imputer drops a column (all NaNs) and we try to assign back to DF
    X_num = X[num_cols].values if len(num_cols) > 0 else None
    X_cat = X[cat_cols].values if len(cat_cols) > 0 else None
    
    # 3. Impute & Process Numeric Data
    if X_num is not None:
        # keep_empty_features=True prevents dropping columns, keeping shapes consistent
        # If your sklearn version is old (<1.2), this param might not exist. 
        # The numpy approach below handles shape changes automatically anyway.
        try:
            imp_num = SimpleImputer(strategy='mean', keep_empty_features=True)
        except TypeError:
            imp_num = SimpleImputer(strategy='mean') # Fallback for older sklearn
            
        X_num = imp_num.fit_transform(X_num)
        
    # 4. Impute & Process Categorical Data
    if X_cat is not None:
        try:
            imp_cat = SimpleImputer(strategy='most_frequent', keep_empty_features=True)
        except TypeError:
            imp_cat = SimpleImputer(strategy='most_frequent')
            
        X_cat = imp_cat.fit_transform(X_cat)
        
        # Ordinal Encode for Tree-based models
        # We cannot use OneHotEncoder for high-dim data (explodes dimensionality)
        X_cat_enc = []
        for i in range(X_cat.shape[1]):
            le = LabelEncoder()
            # Force string conversion to handle mixed types
            col_data = X_cat[:, i].astype(str)
            X_cat_enc.append(le.fit_transform(col_data))
        
        if len(X_cat_enc) > 0:
            X_cat = np.column_stack(X_cat_enc)
        else:
            X_cat = None

    # 5. Concatenate Results
    if X_num is not None and X_cat is not None:
        X_final = np.column_stack([X_num, X_cat])
    elif X_num is not None:
        X_final = X_num
    elif X_cat is not None:
        X_final = X_cat
    else:
        raise ValueError("Dataset resulted in 0 features after processing.")
        
    # 6. Encode Target
    if y is None:
        raise ValueError("Target variable is None.")
    
    # Handle Series or Array
    y = np.array(y)
    y_enc = LabelEncoder().fit_transform(y)
    
    # 7. Final Standardization
    # Good practice for benchmark stability
    scaler = StandardScaler()
    X_final = scaler.fit_transform(X_final)
    
    print(f"Dataset loaded: {X_final.shape[0]} samples, {X_final.shape[1]} features.")
    return X_final, y_enc

# # data_loader.py
# import openml
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import LabelEncoder, StandardScaler
# from sklearn.impute import SimpleImputer

# def load_openml_dataset(dataset_id):
#     """
#     Fetches dataset from OpenML, handles NaN and encoding.
#     Returns: X (numpy array), y (numpy array), name (str)
#     """
#     print(f"Downloading dataset ID {dataset_id}...")
#     dataset = openml.datasets.get_dataset(dataset_id)
#     X, y, categorical_indicator, attribute_names = dataset.get_data(
#         target=dataset.default_target_attribute
#     )

#     # Convert to DataFrame for easier handling initially
#     df = pd.DataFrame(X, columns=attribute_names)
    
#     # 1. Handle Missing Values (Mean for numeric, Mode for categorical)
#     # Simple strategy for benchmarking
#     num_cols = df.select_dtypes(include=[np.number]).columns
#     cat_cols = df.select_dtypes(exclude=[np.number]).columns
    
#     if len(num_cols) > 0:
#         imp_num = SimpleImputer(strategy='mean')
#         df[num_cols] = imp_num.fit_transform(df[num_cols])
        
#     if len(cat_cols) > 0:
#         imp_cat = SimpleImputer(strategy='most_frequent')
#         df[cat_cols] = imp_cat.fit_transform(df[cat_cols])
#         # Encode Categoricals
#         for col in cat_cols:
#             le = LabelEncoder()
#             df[col] = le.fit_transform(df[col].astype(str))

#     # 2. Encode Target
#     y_enc = LabelEncoder().fit_transform(y)
    
#     # 3. Scale Features (Important for many models, though less so for Trees)
#     # We do it to standardize the benchmark environment
#     X_final = StandardScaler().fit_transform(df)
    
#     print(f"Dataset loaded: {X_final.shape[0]} samples, {X_final.shape[1]} features.")
#     return X_final, y_enc

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Author: Nicolas Vana <nicolas.vana@gmail.com>

Inspired by the code and method by: Miron B Kursa, https://m2.icm.edu.pl/boruta/

License: BSD 3 clause
"""

from __future__ import print_function, division
import numpy as np
import scipy as sp
from sklearn.utils import check_random_state, check_X_y
from sklearn.base import TransformerMixin, BaseEstimator, clone
from joblib import Parallel, delayed
import warnings


class GreedyBorutaPyParallel(BaseEstimator, TransformerMixin):
    """
    Greedy Boruta: A faster variant of the Boruta all-relevant feature selection method.
    
    ... [Docstring abbreviated for brevity] ...
    
    Parameters
    ----------
    n_jobs : int, default=1
        The number of jobs to run in parallel. -1 means using all processors.
        
    ... [Other parameters same as before] ...
    """
    def __init__(self, estimator, n_estimators=1000, perc=100, alpha=0.05,
                 two_step=True, n_jobs=1, random_state=None, verbose=0):
        self.estimator = estimator
        self.n_estimators = n_estimators
        self.perc = perc
        self.alpha = alpha
        self.two_step = two_step
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.verbose = verbose
        self.__version__ = '0.4-parallel'
        self._is_lightgbm = 'lightgbm' in str(type(self.estimator))

    def fit(self, X, y):
        return self._fit(X, y)

    def transform(self, X, return_df=False):
        try:
            self.ranking_
        except AttributeError:
            raise ValueError('You need to call the fit(X, y) method first.')

        if return_df:
            X = X.iloc[:, self.support_]
        else:
            X = X[:, self.support_]
        return X

    def fit_transform(self, X, y, return_df=False):
        self._fit(X, y)
        return self.transform(X, return_df)

    def _validate_pandas_input(self, arg):
        try:
            return arg.values
        except AttributeError:
            raise ValueError(
                "input needs to be a numpy array or pandas data frame."
            )

    @staticmethod
    def _parallel_fit_iteration(estimator, X, y, random_state, perc, is_lightgbm):
        """
        Static worker to run one iteration of shadow feature creation and model fitting.
        Returns a boolean array indicating which features were hits.
        """
        # Setup random state for this specific job
        rng = check_random_state(random_state)
        
        # Handle LightGBM specific randomness
        if is_lightgbm:
            estimator.set_params(random_state=rng.randint(0, 10000))
        else:
            estimator.set_params(random_state=rng)

        n_samples, n_features = X.shape
        
        # 1. Create Shadow Features
        # We copy X to create shadows
        x_sha = np.copy(X)
        
        # Shuffle each column individually to destroy relationship with Y
        for i in range(n_features):
            rng.shuffle(x_sha[:, i])
            
        # Ensure we have enough shadows (legacy Boruta behavior: at least 5)
        while x_sha.shape[1] < 5:
            x_sha = np.hstack((x_sha, x_sha))
            
        # 2. Fit Estimator on X + Shadows
        X_combined = np.hstack((X, x_sha))
        
        try:
            estimator.fit(X_combined, y)
        except Exception as e:
            raise ValueError('Estimator fit failed.\n' + str(e))
            
        # 3. Extract Importances
        try:
            imp = estimator.feature_importances_
        except Exception:
            raise ValueError('Estimator must have feature_importances_ attribute.')
            
        # Separate real and shadow importances
        imp_real = imp[:n_features]
        imp_sha = imp[n_features:]
        
        # 4. Determine Hits
        # Calculate shadow threshold
        imp_sha_max = np.percentile(imp_sha, perc)
        
        # Handle NaNs in importance (treat as 0/no-hit)
        cur_imp_no_nan = np.nan_to_num(imp_real)
        
        # A hit is when feature importance > max shadow importance
        hits = (cur_imp_no_nan > imp_sha_max)
        
        return hits

    def _fit(self, X, y):
        # Input validation
        self._check_params(X, y)

        if not isinstance(X, np.ndarray):
            X = self._validate_pandas_input(X) 
        if not isinstance(y, np.ndarray):
            y = self._validate_pandas_input(y)

        self.random_state = check_random_state(self.random_state)
        
        # --- 1. Determine Convergence Criteria ---
        # Since Greedy Boruta has a guaranteed convergence horizon, we calculate K once.
        # No feature is rejected before K, so we don't need a while loop.
        self.max_iter = self._compute_iterations_to_convergence(self.alpha, self.two_step) + 1
        
        n_sample, n_feat = X.shape

        # --- 2. Setup Estimator ---
        # We calculate n_estimators once based on the FULL feature set.
        if self.n_estimators != 'auto':
            self.estimator.set_params(n_estimators=self.n_estimators)
        else:
            n_tree = self._get_tree_num(n_feat)
            self.estimator.set_params(n_estimators=n_tree)

        if self.verbose > 0:
            print(f"Greedy Boruta: Running {self.max_iter} iterations with n_jobs={self.n_jobs}.")

        # --- 3. Parallel Execution ---
        # Generate distinct seeds for reproducibility
        seeds = self.random_state.randint(0, 2**31 - 1, size=self.max_iter)

        # Run K iterations in parallel
        # We use clone(self.estimator) to ensure thread safety
        results = Parallel(n_jobs=self.n_jobs)(
            delayed(self._parallel_fit_iteration)(
                clone(self.estimator), 
                X, 
                y, 
                s, 
                self.perc, 
                self._is_lightgbm
            ) for s in seeds
        )
        
        # Results is a list of boolean arrays (hits)
        # Shape: (n_iterations, n_features)
        all_hits = np.array(results)

        # --- 4. Aggregate Results ---
        # Count how many times each feature was a hit
        total_hits = np.sum(all_hits, axis=0)

        # --- 5. Decision Making ---
        # GREEDY CRITERION:
        # - > 0 hits: Confirmed
        # - 0 hits: Rejected (because p-value for 0 hits in K trials is < alpha)
        
        self.support_ = (total_hits > 0)
        self.n_features_ = np.sum(self.support_)
        
        # Simplified Ranking:
        # 1 = Confirmed
        # 2 = Rejected
        self.ranking_ = np.where(self.support_, 1, 2)
        
        # Note: support_weak_ and importance_history_ are removed as requested
        # because there are no tentative features after K iterations.

        if self.verbose > 0:
            print(f"Finished. Confirmed: {self.n_features_}, Rejected: {n_feat - self.n_features_}")
            
        return self

    def _compute_iterations_to_convergence(self, alpha=0.05, two_step=True):        
        max_iter_to_check = 1000 
        for _iter in range(1, max_iter_to_check + 1):
            p_value = sp.stats.binom.cdf(0, _iter, 0.5)
            threshold = alpha / float(_iter) # Simple Bonferroni
            if p_value <= threshold:
                return _iter
        return None 

    def _get_tree_num(self, n_feat):
        depth = None
        try:
            depth = self.estimator.get_params()['max_depth']
        except KeyError:
            warnings.warn(
                "The estimator does not have a max_depth property, as a result "
                " the number of trees to use cannot be estimated automatically."
            )
        if depth == None:
            depth = 10
        f_repr = 100
        multi = ((n_feat * 2) / (np.sqrt(n_feat * 2) * depth))
        n_estimators = int(multi * f_repr)
        return n_estimators

    def _check_params(self, X, y):
        check_X_y(X, y)
        if self.perc <= 0 or self.perc > 100:
            raise ValueError('The percentile should be between 0 and 100.')
        if self.alpha <= 0 or self.alpha > 1:
            raise ValueError('Alpha should be between 0 and 1.')

In [ ]:
X, y = load_openml_dataset(1485)

